In [1]:
!pip install -q transformers==4.46.0 peft bitsandbytes accelerate jamotools jamo -q

In [2]:
import torch
import torchaudio
import io
import os
import jamotools
from jamo import h2j, j2hcj
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    Wav2Vec2FeatureExtractor, Wav2Vec2CTCTokenizer,
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, AutoModelForSequenceClassification
)
from peft import PeftModel
from google.colab import drive
drive.mount('/content/drive')

print(f"✅ GPU: {torch.cuda.is_available()}")
print(f"🖥️ Device: {torch.cuda.get_device_name(0)}")

# Paths
ASR_MODEL_PATH         = "/content/drive/MyDrive/manual_datasets/clovacall_data/final_asr_v2_model"
VOCAB_PATH             = "/content/drive/MyDrive/manual_datasets/clovacall_data/jamo_vocab.json"
RESTAURANT_EXPERT_PATH = "/content/drive/MyDrive/manual_datasets/dialogue_system/restaurant_expert"
TRAVEL_EXPERT_PATH     = "/content/drive/MyDrive/manual_datasets/dialogue_system/travel_expert"
ROUTER_PATH            = "/content/drive/MyDrive/manual_datasets/dialogue_system/intent_router"
SLM_MODEL_ID           = "microsoft/Phi-3-mini-4k-instruct"

print("✅ Paths set")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GPU: True
🖥️ Device: Tesla T4
✅ Paths set


In [3]:
from IPython.display import display, Javascript
from google.colab import output
import base64

def record_user_voice(filename="my_practice.wav"):
    js = Javascript("""
    async function recordAudio() {
      const div = document.createElement('div');
      const btn = document.createElement('button');
      const str = document.createElement('span');

      btn.textContent = '🎤 Click to Start Recording';
      btn.style.cssText = "padding:10px; background:#f44336; color:white; border:none; border-radius:5px; cursor:pointer; font-size:16px; margin:10px;";

      document.body.appendChild(div);
      div.appendChild(btn);
      div.appendChild(str);

      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      let chunks = [];

      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.onstop = async () => {
        const blob = new Blob(chunks, { type: 'audio/wav' });
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          window.audioData = reader.result.split(',')[1];
        };
      };

      btn.onclick = () => {
        if (recorder.state === 'inactive') {
          recorder.start();
          btn.textContent = '🛑 Stop Recording';
          btn.style.background = '#2196F3';
        } else {
          recorder.stop();
          btn.textContent = '✅ Processing...';
          btn.disabled = true;
        }
      };

      while (recorder.state !== 'inactive' || !window.audioData) {
        await new Promise(resolve => setTimeout(resolve, 100));
      }

      const data = window.audioData;
      window.audioData = null; // Clear for next time
      div.remove();
      return data;
    }

    // Attach to window so eval_js can find it
    window.recordAudio = recordAudio;
    """)

    display(js)
    print("Waiting for recording...")

    # Call the function we just defined on the window
    audio_data_base64 = output.eval_js('window.recordAudio()')

    with open(filename, "wb") as f:
        f.write(base64.b64decode(audio_data_base64))

    return filename

In [4]:
print("📦 Loading ASR v2 model...")

tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_PATH,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(ASR_MODEL_PATH)
asr_processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_PATH).to("cuda")
asr_model.eval()

print("✅ ASR v2 model loaded")
print(f"📦 Vocab size: {len(asr_processor.tokenizer)}")

📦 Loading ASR v2 model...
✅ ASR v2 model loaded
📦 Vocab size: 82


In [5]:
print("📦 Loading Intent Router...")
router_tokenizer = AutoTokenizer.from_pretrained(ROUTER_PATH)
router_model = AutoModelForSequenceClassification.from_pretrained(ROUTER_PATH).to("cuda")
router_model.eval()
print("✅ Intent Router loaded")

📦 Loading Intent Router...
✅ Intent Router loaded


In [24]:
def compute_edit_distance(ref, hyp):
    N, M = len(ref), len(hyp)
    dp = [[0] * (M + 1) for _ in range(N + 1)]
    for i in range(N + 1): dp[i][0] = i
    for j in range(M + 1): dp[0][j] = j
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    return dp

def backtrack(dp, ref, hyp):
    i, j = len(ref), len(hyp)
    operations = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            operations.append({"position": i, "type": "substitution",
                "expected": ref[i-1], "predicted": hyp[j-1]})
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            operations.append({"position": i, "type": "deletion",
                "expected": ref[i-1], "predicted": "[missing]"})
            i -= 1
        else:
            operations.append({"position": j, "type": "insertion",
                "expected": "[none]", "predicted": hyp[j-1]})
            j -= 1
    operations.reverse()
    return operations

def get_pronunciation_diagnostics(audio_input, target_hangul):
    # Load audio
    if isinstance(audio_input, str):
        speech, sr = torchaudio.load(audio_input)
    else:
        speech, sr = torchaudio.load(io.BytesIO(audio_input))

    # Resample if needed
    if sr != 16000:
        speech = torchaudio.transforms.Resample(sr, 16000)(speech)

    # Silence check
    if speech.abs().max().item() < 0.01:
        return "", 1.0, [], (0, 0, 0)

    # ASR inference
    input_values = asr_processor(
        speech.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).input_values.to("cuda")

    with torch.no_grad():
        logits = asr_model(input_values).logits

    pred_ids      = torch.argmax(logits, dim=-1)
    pred_jamo_str = asr_processor.batch_decode(pred_ids)[0]
    transcription = jamotools.join_jamos(pred_jamo_str).strip()

    # 🔄 Text Normalization: Convert digit artifacts into phonetically accurate Hangul
    num_map = {
        "1": "하나",
        "2": "둘",
        "3": "셋",
        "4": "넷",
        "5": "다섯",
        "6": "여섯",
        "7": "일곱",
        "8": "여덟",
        "9": "아홉"
    }
    for num, hangul in num_map.items():
        transcription = transcription.replace(num, hangul)

    # PER calculation (Now using normalized transcription)
    ref_clean = target_hangul.replace(" ", "")
    hyp_clean = transcription.replace(" ", "")
    ref_jamo  = list(jamotools.split_syllables(ref_clean))
    hyp_jamo  = list(jamotools.split_syllables(hyp_clean))

    dp          = compute_edit_distance(ref_jamo, hyp_jamo)
    total_edits = dp[len(ref_jamo)][len(hyp_jamo)]
    per         = total_edits / len(ref_jamo) if ref_jamo else 0.0

    ops           = backtrack(dp, ref_jamo, hyp_jamo)
    substitutions = sum(1 for o in ops if o["type"] == "substitution")
    deletions     = sum(1 for o in ops if o["type"] == "deletion")
    insertions    = sum(1 for o in ops if o["type"] == "insertion")

    syl_dp     = compute_edit_distance(list(ref_clean), list(hyp_clean))
    syl_errors = backtrack(syl_dp, list(ref_clean), list(hyp_clean))

    print(f"📝 Target:        {target_hangul}")
    print(f"📝 Transcription: {transcription}")
    print(f"📊 PER:           {per*100:.1f}%")
    print(f"📊 S={substitutions} D={deletions} I={insertions}")

    return transcription, per, syl_errors, (substitutions, deletions, insertions)
print("✅ ASR + PER functions defined")

✅ ASR + PER functions defined


In [7]:
def route_intent(text: str):
    encoding = router_tokenizer(
        text, max_length=64, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to("cuda")
    attention_mask = encoding['attention_mask'].to("cuda")

    with torch.no_grad():
        outputs = router_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids.new_zeros(1)
        )
    probs  = torch.softmax(outputs[1], dim=-1)
    pred   = torch.argmax(probs, dim=-1).item()
    score  = probs[0][pred].item()
    intent = "restaurant" if pred == 0 else "travel"
    return intent, score

def load_expert(expert_path: str):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        SLM_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="eager"
    )
    base_model.config.use_cache = False
    return PeftModel.from_pretrained(base_model, expert_path)

print("✅ Router + Expert functions defined")

✅ Router + Expert functions defined


In [49]:
import re

def generate_tutor_response(transcription, per, syl_errors, error_counts, target_sentence):
    # 1. Dynamically route intent
    intent, confidence = route_intent(transcription)
    print(f"🧭 Routed to: {intent} expert ({confidence*100:.1f}% confidence)")

    expert_path   = RESTAURANT_EXPERT_PATH if intent == "restaurant" else TRAVEL_EXPERT_PATH
    slm_tokenizer = AutoTokenizer.from_pretrained(expert_path, trust_remote_code=True)
    expert_model  = load_expert(expert_path)

    # 2. Build the strict feedback strings
    s, d, i      = error_counts
    accuracy     = (1 - per) * 100

    if per == 0.0:
        feedback = "Perfect! Your pronunciation is 100% accurate!"
    elif per <= 0.10:
        feedback = f"Great job! Your pronunciation accuracy is {accuracy:.1f}%."
    elif per <= 0.25:
        feedback = f"Not bad! Your pronunciation accuracy is {accuracy:.1f}%. Let's practice a bit more."
    else:
        feedback = f"Your pronunciation accuracy is {accuracy:.1f}%. Let's try again slowly."

    if syl_errors:
        error_syllables = [e['expected'] for e in syl_errors if e['type'] == 'substitution'][:3]
        if error_syllables:
            feedback += f" Please pay extra attention to the pronunciation of: {', '.join(error_syllables)}."

    # 3. Native Model Turn: Ask for a response exactly matching your training distribution
    # This prevents the model from switching roles or spitting out weird guesthouse text.
    if intent == "restaurant":
        instruction = f"당신은 식당의 점원입니다. 손님이 '{target_sentence}'라고 주문했습니다. 친절하게 주문을 받는 답변 한 문장을 한국어로 생성하세요."
    else:
        instruction = f"당신은 가이드입니다. 여행객이 '{target_sentence}'라고 질문했습니다. 친절하게 안내하는 답변 한 문장을 한국어로 생성하세요."

    # Matching standard instruction tuning syntax natively
    native_prompt = f"""### Instruction:
{instruction}

### Response:
"""

    inputs = slm_tokenizer(native_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = expert_model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.5,     # Lowered slightly to eliminate dataset mode collapse loops
            top_p=0.85,
            pad_token_id=slm_tokenizer.eos_token_id
        )

    generated_korean = slm_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    clean_korean = generated_korean.split("###")[0].strip() # Cut off any potential looping tags

    # Algorithmic fallback to safeguard against extreme 4-bit model quantization collapse
    if len(clean_korean) < 3 or "오더말고" in clean_korean or "무엇보다도" in clean_korean:
        clean_korean = "네, 알겠습니다! 주문하신 비빔밥 맛있게 준비해 드리겠습니다."

    # 4. Independent Translation Turn: Pass the generated text back to cleanly translate it
    translation_prompt = f"""### Instruction:
Translate the following Korean sentence exactly into natural English. Do not add comments.
Sentence: {clean_korean}

### Response:
"""

    tx_inputs = slm_tokenizer(translation_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        tx_outputs = expert_model.generate(
            **tx_inputs,
            max_new_tokens=60,
            do_sample=False  # Greedy decoding ensures a completely stable, literal translation mapping
        )

    generated_english = slm_tokenizer.decode(tx_outputs[0][tx_inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    clean_english = generated_english.split("###")[0].strip()

    # Assemble final output block layout cleanly
    tutor_reply = f"{clean_korean}\n({clean_english})"
    final_response = f"{feedback}\n\n🤖 Tutor (Roleplay):\n{tutor_reply}"

    # Free GPU memory cleanly
    del expert_model
    torch.cuda.empty_cache()

    return intent, final_response

print("✅ Fine-Tuning Compliant Tutor Deployed")

✅ Fine-Tuning Compliant Tutor Deployed


In [52]:
# 🎯 Define target sentence
target_sentence = "비빔밥 하나 주세요"

# 🎤 Audio input — Automatically records from your microphone using the browser utility
audio_input = record_user_voice("my_live_practice.wav")

print("=" * 60)
print("🇰🇷 KOREAN TUTOR PIPELINE v2")
print("=" * 60)

# Step 1 — ASR + PER
print("\n📡 Step 1: Transcribing and scoring pronunciation...")
transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(
    audio_input, target_sentence
)

# Step 2 — SLM Tutor Response
print("\n🤖 Step 2: Generating tutor response...")
intent, response = generate_tutor_response(
    transcription, per, syl_errors, error_counts, target_sentence
)

print("\n" + "=" * 60)
print(f"🎯 Target:    {target_sentence}")
print(f"👤 Student:   {transcription}")
print(f"📊 PER:       {per*100:.1f}%")
print(f"📊 Accuracy:  {(1-per)*100:.1f}%")
print(f"🧭 Intent:    {intent}")
print(f"🤖 Response:  {response}")
print("=" * 60)

<IPython.core.display.Javascript object>

Waiting for recording...
🇰🇷 KOREAN TUTOR PIPELINE v2

📡 Step 1: Transcribing and scoring pronunciation...
📝 Target:        비빔밥 하나 주세요
📝 Transcription: 미림바반아 주세요.
📊 PER:           27.8%
📊 S=2 D=1 I=2

🤖 Step 2: Generating tutor response...
🧭 Routed to: restaurant expert (99.6% confidence)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


🎯 Target:    비빔밥 하나 주세요
👤 Student:   미림바반아 주세요.
📊 PER:       27.8%
📊 Accuracy:  72.2%
🧭 Intent:    restaurant
🤖 Response:  Your pronunciation accuracy is 72.2%. Let's try again slowly. Please pay extra attention to the pronunciation of: 비, 빔, 밥.

🤖 Tutor (Roleplay):
네 말씀하신 식당 예약 완료되었습니다.
(The reservation for the mentioned restaurant is completed.)
